# 04 - Price Prediction

## Doel
Goudprijzen 2 maanden vooruit voorspellen met minstens 2 modellen.

## Modellen
| Model | Type | Waarom? |
|-------|------|--------|
| **Linear Regression** | Baseline | Interpreteerbaar, goed als referentie |
| **ARIMA** | Time series | Vangt temporele patronen en autocorrelatie |
| **Naive (Random Walk)** | Baseline | Simpelste voorspelling; must to beat |

## Eerlijke kanttekeningen
- Financiele tijdreeksen zijn moeilijk te voorspellen (lage signaal/ruis ratio)
- ARIMA kan slechter presteren dan een naive model op financiele data
- De modellen zijn **directionele indicatoren**, niet kristalbollen
- Scenario-analyse (Notebook 05) completeert de voorspelling

In [ ]:
# Cel 1: Inladen van bibliotheken en instellen van paden

import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.predictor import (
    create_lag_features, train_test_split_time,
    train_linear_regression, train_arima, auto_arima_order,
    naive_forecast, compare_models, save_model
)
from src.data_loader import load_raw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
sns.set_palette('husl')

PROC_DIR = Path('../data/processed')
TROY_OZ_TO_GRAM = 31.1035
print('Setup complete.')

In [ ]:
# Cel 2: Schone data en features inladen voor modeltraining

# Laad schone data
gold = pd.read_csv(PROC_DIR / 'gold_with_macro.csv', index_col=0, parse_dates=True)
gold.index = pd.to_datetime(gold.index)

print(f'Data: {len(gold)} observaties van {gold.index.min().date()} tot {gold.index.max().date()}')
print(f'Kolommen: {list(gold.columns)}')
gold.tail()

## 1. Data Voorbereiding

We voorspellen de goudprijs **2 maanden (40 handelsdagen)** vooruit.

In [ ]:
# Cel 3: Doelvariabele aanmaken: goudprijs 40 handelsdagen vooruit

# Doelvariabele: goudprijs 40 dagen vooruit
TARGET = 'gold_eur_gram'
HORIZON = 40  # 2 maanden = ~40 handelsdagen

# Features: lagged prices + macro
df = gold.copy()
df['target'] = df[TARGET].shift(-HORIZON)  # Prijs 40 dagen in de toekomst

# Lag features toevoegen
lags = [1, 2, 5, 10, 20, 40]
for lag in lags:
    df[f'lag_{lag}'] = df[TARGET].shift(lag)

# Rendementen toevoegen
df['return_5d'] = df[TARGET].pct_change(5)
df['return_20d'] = df[TARGET].pct_change(20)
df['volatility_20d'] = df[TARGET].pct_change().rolling(20).std()

# EUR/USD als feature
if 'eur_usd' in df.columns:
    df['eur_usd_lag5'] = df['eur_usd'].shift(5)

# Rijen met NaN verwijderen
df = df.dropna()
print(f'Data na feature engineering: {len(df)} observaties')
print(f'Doelvariabele: target (goudprijs over {HORIZON} dagen)')

In [ ]:
# Cel 4: Tijdreeks-bewuste train/test split aanmaken

# Train/test split (time-series aware)
feature_cols = [c for c in df.columns if c not in ['target', 'daily_return'] and df[c].dtype in ['float64', 'int64']]
X = df[feature_cols]
y = df['target']

train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

print(f'Training set: {len(X_train)} observaties ({X_train.index[0].date()} tot {X_train.index[-1].date()})')
print(f'Test set:     {len(X_test)} observaties ({X_test.index[0].date()} tot {X_test.index[-1].date()})')

# Visuele check van split
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_train.index, y_train, label='Training', color='blue')
ax.plot(y_test.index, y_test, label='Test', color='red')
ax.axvline(y_test.index[0], color='black', linestyle='--', label='Train/Test split')
ax.set_title('Train/Test Split (Time-Series)')
ax.set_ylabel('Goudprijs (EUR/gram)')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Naive Baseline (Random Walk)

De simpelste voorspelling: morgen = vandaag. Elk serieus model moet dit verslaan.

In [ ]:
# Cel 5: Naive baseline model trainen: morgen = vandaag (random walk)

naive_metrics, y_pred_naive = naive_forecast(y_train, y_test)
print('--- Naive (Random Walk) ---')
for k, v in naive_metrics.items():
    if k != 'model_name':
        print(f'  {k}: {v:.4f}' if not pd.isna(v) else f'  {k}: N/A')

## 3. Model A: Linear Regression

### Waarom Linear Regression?
- **Interpreteerbaar**: Elk co-efficient heeft een betekenis
- **Baseline**: Vestigt een minimum prestatieniveau
- **Snel**: Geen complexe hyperparameter tuning nodig

### Begrijpbaar uitleg voor niet-technisch publiek:
"Stel je voor dat je een rechte lijn trekt door alle vorige goudprijzen. Die lijn
geeft aan of goud gemiddeld stijgt of daalt. We gebruiken die lijn om te voorspellen
waar de prijs over 2 maanden zou kunnen zijn."

In [ ]:
# Cel 6: Linear Regression model trainen en evalueren

lr_model, lr_metrics, lr_importance, y_pred_lr = train_linear_regression(
    X_train, y_train, X_test, y_test
)

print('--- Linear Regression ---')
for k, v in lr_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\n--- Feature Importance ---')
print(lr_importance.head(10).to_string(index=False))

In [ ]:
# Cel 7: Feature importance van Linear Regression visualiseren

# Feature importance visualisatie
fig, ax = plt.subplots(figsize=(10, 6))
top_features = lr_importance.head(12)
colors = ['green' if c > 0 else 'red' for c in top_features['coefficient']]
ax.barh(top_features['feature'], top_features['coefficient'], color=colors)
ax.set_title('Top 12 Feature Coefficients (Linear Regression)')
ax.set_xlabel('Coefficient (impact op goudprijs)')
ax.axvline(0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 4. Model B: ARIMA

### Waarom ARIMA?
- **Time-series specifiek**: Ontworpen voor tijdreeks data
- **Autocorrelatie**: Vangt patronen in hoe heden samenhangt met verleden
- **Automatische parameter keuze**: Gebruikt AIC voor objectieve selectie

### ARIMA Parameters:
- **AR (p)**: Autoregressie - hoeveel vorige waarden bepalen de huidige
- **I (d)**: Integratie - hoeveel keer differentieren om stationair te maken
- **MA (q)**: Moving average - hoeveel voorspellingsfouten worden meegenomen

### Begrijpbaar uitleg:
"ARIMA kijkt niet naar de goudprijs op zich, maar naar de **patronen** in de prijs.
Als goud de afgelopen week met 2% is gestegen, en historisch gezien een stijging
vaak wordt gevolgd door verdere stijging, dan houdt ARIMA daar rekening mee."

In [ ]:
# Cel 8: Beste ARIMA orde zoeken via grid search op AIC

# Auto ARIMA: beste orde zoeken
print('Zoek beste ARIMA orde (kan even duren)...')
best_order, best_aic = auto_arima_order(y_train, max_p=3, max_d=2, max_q=3)

In [ ]:
# Cel 9: ARIMA model trainen met de beste orde

# Train ARIMA met beste orde
arima_model, arima_metrics, y_pred_arima = train_arima(
    y_train, y_test, order=best_order
)

print(f'\n--- ARIMA{best_order} ---')
for k, v in arima_metrics.items():
    if k != 'model_name':
        print(f'  {k}: {v:.4f}' if not pd.isna(v) else f'  {k}: N/A')

In [ ]:
# Cel 10: ARIMA residuen analyseren en diagnosticeren

# ARIMA residuen diagnose
residuen = arima_model.resid

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residuen plot
axes[0].plot(residuen, color='blue', alpha=0.6)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title(f'ARIMA{best_order} Residuen')
axes[0].set_ylabel('Residu')

# Histogram residuen
axes[1].hist(residuen, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_title('Verdeling Residuen')
axes[1].axvline(0, color='red', linestyle='--')

# ACF van residuen
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(residuen.dropna(), ax=axes[2], lags=20)
axes[2].set_title('Autocorrelatie Residuen')

plt.tight_layout()
plt.show()

print(f'Gemiddelde residu: {residuen.mean():.4f} (moet dicht bij 0 liggen)')
print(f'Std residu: {residuen.std():.4f}')

## 5. Model Vergelijking

In [ ]:
# Cel 11: Modellen vergelijken op RMSE, MAE en R-kwadraat

# Vergelijking
all_metrics = [naive_metrics, lr_metrics, arima_metrics]
comparison = compare_models(all_metrics)

print('--- MODEL VERGELIJKING ---')
print(comparison.to_string())

# Visualisatie
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics_to_plot = ['test_rmse', 'test_mae', 'test_r2']
titles = ['Test RMSE (lager = beter)', 'Test MAE (lager = beter)', 'Test R-squared (hoger = beter)']

for i, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
    vals = comparison[metric].dropna()
    colors = ['green' if 'Naive' in n else 'steelblue' for n in vals.index]
    if metric == 'test_r2':
        colors = ['green' if 'Naive' in n else 'steelblue' for n in vals.index]
    axes[i].bar(range(len(vals)), vals.values, color=colors)
    axes[i].set_xticks(range(len(vals)))
    axes[i].set_xticklabels(vals.index, rotation=15, fontsize=9)
    axes[i].set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
# Cel 12: Voorspellingen van alle modellen visualiseren

# Voorspellingen visualiseren
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(y_test.index, y_test, label='Werkelijke prijs', color='black', linewidth=2)
ax.plot(y_test.index, y_pred_lr, label='Linear Regression', color='blue', linestyle='--', alpha=0.7)
ax.plot(y_test.index, y_pred_arima, label=f'ARIMA{best_order}', color='red', linestyle='--', alpha=0.7)
ax.plot(y_test.index, y_pred_naive, label='Naive (Random Walk)', color='green', linestyle=':', alpha=0.7)

ax.set_title(f'Voorspellingen vs Werkelijkheid ({HORIZON} dagen horizon)', fontsize=14)
ax.set_ylabel('Goudprijs (EUR/gram)')
ax.set_xlabel('Datum')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Model Evaluatie & Beperkingen

### Eerlijke beoordeling:

**Wat de modellen goed doen:**
- Baseline vergelijking (naive vs. modellen)
- Richting van de trend vangen

**Wat de modellen NIET kunnen:**
- Externe schokken voorspellen (geopolitieke gebeurtenissen, rentewijzigingen)
- Structurele veranderingen in de markt
- Garantie geven op de voorspelling

**Beperkingen:**
1. **Kleine dataset**: Beperkt aantal aankopen om tegen te valideren
2. **Lage R-squared**: Financiele tijdreeksen hebben inherent lage voorspelbaarheid
3. **Overfitting risico**: Zelfs met time-series cross-validation
4. **Regime veranderingen**: Modellen gaan uit van stabiele patronen

### Wat als het model ongelijk heeft?
> "De modellen moeten worden gezien als **directionele indicatoren**, niet als
kristalbollen. De echte waarde zit in gedisciplineerde besluitvorming:
het forceert om aannames te kwantificeren en tegen het licht te houden."

Daarom is **scenario-analyse** (Notebook 05) essentieel als aanvulling.

In [ ]:
# Cel 13: Toekomstige goudprijs voorspellen (2 maanden vooruit)

# Laatste voorspelling (2 maanden vooruit)
laatste_prijs = gold[TARGET].iloc[-1]
laatste_features = X.iloc[-1:].values

voorspelling_lr = lr_model.predict(laatste_features)[0]
voorspelling_arima = arima_model.forecast(steps=1).iloc[0]

print('=' * 50)
print('   LAATSTE VOORSPELLING (2 MAANDEN VOORUIT)')
print('=' * 50)
print(f'Huidige prijs:     EUR {laatste_prijs:.2f}/gram')
print(f'                  EUR {laatste_prijs * 10:.2f} voor 10g bar')
print()
print(f'Linear Reg.:       EUR {voorspelling_lr:.2f}/gram ({(voorspelling_lr/laatste_prijs-1)*100:+.1f}%)')
print(f'                  EUR {voorspelling_lr * 10:.2f} voor 10g bar')
print()
print(f'ARIMA{best_order}:        EUR {voorspelling_arima:.2f}/gram ({(voorspelling_arima/laatste_prijs-1)*100:+.1f}%)')
print(f'                  EUR {voorspelling_arima * 10:.2f} voor 10g bar')
print()
print('Disclaimer: Dit is een richtwaarde, geen garantie.')
print('Gebruik samen met scenario-analyse (Notebook 05).')
print('=' * 50)

In [ ]:
# Cel 14: Getrainde modellen opslaan naar disk

# Modellen opslaan
save_model(lr_model, 'linear_regression')
save_model(arima_model, 'arima_model')
print('\nModellen opgeslagen in data/models/')

---
## Samenvatting

| Model | Test RMSE | Test MAE | Test R2 | Notitie |
|-------|-----------|----------|---------|--------|
| Naive | TBD | TBD | TBD | Baseline |
| Linear Regression | TBD | TBD | TBD | Interpreteerbaar |
| ARIMA | TBD | TBD | TBD | Time-series specifiek |

**Volgende stap:** Notebook 05 - Scenario Analyse